Tensorflow and Keras were installed following the official tutorial:
https://www.tensorflow.org/install/pip

In [ ]:
import tensorflow as tf
from tensorflow import keras

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import sys
sys.path.append("..")
import methods.explainability as ex
from methods import utils

physical_devices = tf.config.list_physical_devices("GPU")
print("Num GPUs:", len(physical_devices))

In [ ]:
NOTEBOOK = "2D_keras_binary_L"
NORM = "none"
RES = 0.05
plane = "XY"
DIMS = "2D"
k_cv = 5
IMGS_DIR = f"../../data/preprocessed/2D_res={RES}_norm={NORM}_{k_cv}fold_withDMSO"

CNN_DENSE_FILTS = (256, 64, 16)
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 64
CHANNEL_MODE = "grayscale"
N_OUTPUT_UNITS = 1
LR_SCHED = None
OPTIMIZER = "AdamW"
COMMENT = "testing baseline"

EPOCHS = 50
LR = 1e-4
SEED = 2023

if CHANNEL_MODE == "rgb":
    channels = (3,)
elif CHANNEL_MODE == "grayscale":
    channels = (1,)

if N_OUTPUT_UNITS == 1:
    OUTPUT_FUNC = "sigmoid"
    LOSS_FUNC = tf.keras.losses.BinaryCrossentropy(
        label_smoothing=0.1,
    )
    LABEL_MODE = "binary"
elif N_OUTPUT_UNITS == 2:
    OUTPUT_FUNC = "softmax"
    LOSS_FUNC = "categorical_crossentropy"
    LABEL_MODE = "categorical"

In [ ]:
sns.color_palette("colorblind")
pal = utils.get_class_palette()
mic_pal = utils.get_microscopist_palette()

In [ ]:
k = 1 
train_ds = None

for i in range(1, k_cv + 1):
    if i == k:
        val_ds = tf.keras.utils.image_dataset_from_directory(
            f"{IMGS_DIR}/fold_{i}/",
            color_mode=CHANNEL_MODE,
            labels="inferred",
            label_mode=LABEL_MODE,
            interpolation="bilinear",
            seed=SEED,
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            shuffle=False,
        )
        continue

    if train_ds:
        train_ds = train_ds.concatenate(
            tf.keras.utils.image_dataset_from_directory(
                f"{IMGS_DIR}/fold_{i}/",
                color_mode=CHANNEL_MODE,
                labels="inferred",
                label_mode=LABEL_MODE,
                interpolation="bilinear",
                seed=SEED,
                image_size=IMAGE_SIZE,
                batch_size=BATCH_SIZE,
                shuffle=True,
            )
        )
    else:
        train_ds = tf.keras.utils.image_dataset_from_directory(
            f"{IMGS_DIR}/fold_{i}/",
            color_mode=CHANNEL_MODE,
            labels="inferred",
            label_mode=LABEL_MODE,
            interpolation="bilinear",
            seed=SEED,
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            shuffle=True,
        )

n_train_ims = train_ds.cardinality().numpy() * BATCH_SIZE
print(n_train_ims)

train_ds = train_ds.unbatch().shuffle(10000).batch(BATCH_SIZE)
# Prefetching samples in GPU memory helps maximize GPU utilization.
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)

In [ ]:
model = keras.models.load_model("../models/chromagenet/calmodel_fold_1.keras")
model.load_weights("../models/checkpoints/fold_1/44-0.692.weights.h5")

lr_obj = LR

if OPTIMIZER == "AdamW":
    optim = keras.optimizers.AdamW(
        learning_rate=lr_obj, use_ema=True
    )
elif OPTIMIZER == "Adam":
    optim = keras.optimizers.Adam(learning_rate=lr_obj, use_ema=True)
elif OPTIMIZER == "SGD":
    optim = keras.optimizers.SGD(learning_rate=lr_obj, momentum=0.6)

metrics = [
    "accuracy",
    tf.keras.metrics.AUC(),
    tf.keras.metrics.Precision(),
    tf.keras.metrics.Recall(),
    tf.keras.metrics.F1Score(),
]

model.compile(
    optimizer=optim,
    loss=LOSS_FUNC,
    metrics=metrics,
    jit_compile=True,
)

In [ ]:
model.evaluate(val_ds)

In [ ]:
model.summary()

In [ ]:
train_ids, train_ds = utils.dataset_from_partition_cv(
    IMGS_DIR, ["fold_2", "fold_3", "fold_4", "fold_5"], CHANNEL_MODE, 
    LABEL_MODE, IMAGE_SIZE, BATCH_SIZE, SEED
)
val_ids, val_ds = utils.dataset_from_partition(
    IMGS_DIR,
    "fold_1",
    CHANNEL_MODE,
    LABEL_MODE,
    IMAGE_SIZE,
    BATCH_SIZE,
    SEED,
)

In [ ]:
train_df = ex.get_output_df_voting(train_ids, train_ds, model, LABEL_MODE)
train_df["class"] = train_df["label"].map({0: 'aged', 1: 'young'})
val_df = ex.get_output_df_voting(val_ids, val_ds, model, LABEL_MODE)
val_df["class"] = val_df["label"].map({0: 'aged', 1: 'young'})
# test_df = ex.get_output_df_voting(test_ids, test_ds, model, LABEL_MODE)

In [ ]:
val_df

In [ ]:
dataset_names = ["train", "val"]
dataframes = [train_df, val_df]
thresholds = np.arange(0.3, 0.8, 0.05)

thresh_df = ex.plot_metrics(dataframes, dataset_names, thresholds)

for m_name in ["acc", "precision", "recall", "f1"]:
    sns.lineplot(x="threshold", y=m_name, hue="dataset", data=thresh_df)
    plt.title(m_name)
    plt.show()

In [ ]:
plt.figure(figsize=(4, 3))
sns.displot(train_df, x="mean_prob", hue="class", binwidth=0.05, kde=True, palette=pal)
plt.xlim(0, 1)
ex.plot_conf_mat(train_df["label"], train_df["mean_prob"], threshold=0.45)
ex.plot_auc(train_df["label"], train_df["mean_prob"], threshold=0.45)

In [ ]:
plt.figure(figsize=(4, 3))
sns.displot(val_df, x="mean_prob", hue="class", binwidth=0.05, kde=True, palette=pal)
plt.xlabel("Averaged probability")
plt.ylabel("Number of nuclei")
plt.xlim(0, 1)
ex.plot_conf_mat(val_df["label"], val_df["mean_prob"], threshold=0.45)
ex.plot_auc(val_df["label"], val_df["mean_prob"], threshold=0.45)

### Compounds without model calibration

In [ ]:
conditions = [
    "aged_CASIN",
    "aged_CDK8i",
    #"aged_DMSO",
    "aged_IOX",
    "aged_UNC",
    "aged_treated_RhoAi",
]

In [ ]:
col_names = ["nuc_id", "mean_prob", "median_prob", "std_prob", "hard_preds", "condition"]
cond_df = pd.DataFrame(columns=col_names)

for cond in conditions:
    print(f"Processing condition {cond}")
    cond_ids, cond_ds = utils.dataset_from_partition(
        imgs_dir="../../data/preprocessed/2D_res=0.05_norm=none_XY_final",
        partition=cond,
        channel_mode=CHANNEL_MODE,
        label_mode=None,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        seed=SEED,
    )
    this_df = ex.predict_df_voting(cond_ids, cond_ds, model, LABEL_MODE)
    this_df["condition"] = cond
    cond_df = pd.concat([cond_df, this_df])

cond_df

In [ ]:
val_df["condition"] = np.where(val_df["label"] == 1, "young", "aged")
val_df.drop(["label", "class"], axis=1, inplace=True)
cond_df = pd.concat([cond_df, val_df])
cond_df

In [ ]:
for cond in conditions:
    sns.displot(cond_df[cond_df["condition"] == cond], x="mean_prob", 
                binwidth=0.05, height=2, color=pal[cond])
    plt.title(cond)
    plt.xlim(0, 1)
    plt.show()

for cond in ["young", "aged"]:
    sns.displot(cond_df[cond_df["condition"] == cond], x="mean_prob", 
                binwidth=0.05, height=2, color=pal[cond])
    plt.title(cond)
    plt.xlim(0, 1)
    plt.show()

In [ ]:
grouped = cond_df.groupby("condition")["mean_prob"]

summary_df = grouped.agg(
    mean="mean",
    std="std",
    count="count"
).reset_index()

summary_df["CI"] = summary_df.apply(
    lambda row: utils.confidence_interval(row["count"], row["mean"], row["std"]), axis=1
)

summary_df

### With Beta Calibration

In [ ]:
from betacal import BetaCalibration

bc = BetaCalibration(parameters="abm")
uncal_probs = model.predict(val_ds)
val_labels = np.concatenate([y for _, y in val_ds], axis=0)

bc.fit(uncal_probs.ravel(), val_labels.ravel())

cal_val_df = ex.get_output_df_voting(val_ids, val_ds, model, 
                                     LABEL_MODE, thresh=0.45, 
                                     cal_model=bc)
cal_val_df["class"] = cal_val_df["label"].map({0: 'aged', 1: 'young'})

In [ ]:
conditions = [
    "aged_DMSO",
    "aged_CASIN",
    "aged_IOX",
    "aged_UNC",
    "aged_treated_RhoAi",
]

col_names = ["nuc_id", "mean_prob", "median_prob", "std_prob", "hard_preds", "condition"]
cond_df = pd.DataFrame(columns=col_names)

for cond in conditions:
    print(f"Processing condition {cond}")
    cond_ids, cond_ds = utils.dataset_from_partition(
        imgs_dir="../../data/preprocessed/2D_res=0.05_norm=none_XY_final",
        partition=cond,
        channel_mode=CHANNEL_MODE,
        label_mode=None,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        seed=SEED,
    )
    this_df = ex.predict_df_voting(cond_ids, cond_ds, model, 
                                   LABEL_MODE, thresh=0.45, cal_model=bc)
    this_df["condition"] = cond
    cond_df = pd.concat([cond_df, this_df])

cond_df

In [ ]:
cal_val_df["condition"] = np.where(cal_val_df["label"] == 1, "young", "aged")
cal_val_df.drop(["label", "class"], axis=1, inplace=True)
cond_df = pd.concat([cond_df, cal_val_df])
cond_df["condition"] = cond_df["condition"].replace("aged_treated_RhoAi", "aged_RhoAi")
cond_df

In [ ]:
# Merge aged and aged + DMSO
cond_df["condition"] = cond_df["condition"].replace("aged_DMSO", "aged")

In [ ]:
conditions = [
    "aged",
    "aged_CASIN",
    "aged_IOX",
    "aged_UNC",
    "aged_RhoAi",
    "young",
]

# Only include specified conditions
cond_df = cond_df[cond_df["condition"].isin(conditions)]

colors = sns.color_palette("colorblind")
colors2 = sns.color_palette("tab20")
pal = {
    "young": colors[2],
    "aged": colors[3],
    "aged_CASIN": colors[0],
    "aged_DMSO": colors[4],
    "aged_IOX": colors[5],
    "aged_UNC": colors[6],
    "aged_RhoAi": colors[7],
}

In [ ]:
for cond in conditions:
    sns.displot(cond_df[cond_df["condition"] == cond], x="mean_prob", 
                binwidth=0.05, height=2, color=pal[cond])
    plt.title(cond)
    plt.xlim(0, 1)
    plt.show()

for cond in ["young", "aged"]:
    sns.displot(cond_df[cond_df["condition"] == cond], x="mean_prob", 
                binwidth=0.05, height=2, color=pal[cond])
    plt.title(cond)
    plt.xlim(0, 1)
    plt.show()

In [ ]:
grouped = cond_df.groupby("condition")["mean_prob"]

summary_df = grouped.agg(
    mean="mean",
    std="std",
    median="median",
    count="count"
).reset_index()

summary_df["CI"] = summary_df.apply(
    lambda row: utils.confidence_interval(row["count"], row["mean"], row["std"]), axis=1
)

summary_df

In [ ]:
summary_df.to_latex(float_format="%.3f")

In [ ]:
f, ax = plt.subplots(figsize=(8, 3))
sns.boxplot(
    cond_df,
    x="condition",
    y="mean_prob",
    palette=pal,
)
ax.set_ylabel("Predicted Probability")
ax.set_xlabel("HSC Condition")
plt.xticks(rotation=90)

In [ ]:
utils.plot_distribution_comparison(cond_df, pal)